In [3]:
!pip install diffusers

In [4]:
import torch
from torch.utils.data import DataLoader,Dataset
from torchvision.transforms import transforms
from diffusers import UNet2DConditionModel, DDPMScheduler, AutoencoderKL
from transformers import CLIPTokenizer, CLIPTextModel
from tqdm.auto import tqdm
from PIL import Image
import os
import json

print(torch.__version__)

2.8.0+cu128


# Dataset

In [5]:
from datasets import load_dataset
from torch.utils.data import IterableDataset
from torchvision import transforms
from PIL import Image
import torch
import requests
from io import BytesIO

# Load streaming dataset
hf_dataset = load_dataset(
    "laion/relaion400m",
    split="train",
    streaming=True
)

# Filter NSFW + similarity
hf_dataset = hf_dataset.filter(
    lambda x: x["NSFW"] == "UNLIKELY" and x["similarity"] > 0.28
)

Resolving data files:   0%|          | 0/128 [00:00<?, ?it/s]

In [6]:
class LAIONStreamDataset(IterableDataset):
    def __init__(self, hf_dataset, image_size=64):
        self.dataset = hf_dataset

        self.transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def download_image(self, url, timeout=5):
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")
        return image

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()

        if worker_info is None:
            dataset_iter = iter(self.dataset)
        else:
            dataset_iter = iter(
                self.dataset.shard(
                    num_shards=worker_info.num_workers,
                    index=worker_info.id
                )
            )

        for sample in dataset_iter:
            try:
                image = self.download_image(sample["url"])
                image = self.transform(image)
                caption = sample["caption"]

                yield image, caption
            except:
                continue

In [7]:
from torch.utils.data import DataLoader

train_dataset = LAIONStreamDataset(hf_dataset, image_size=64)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,          # 64 works well at 64x64 on L4
    num_workers=4,
    pin_memory=True,
    prefetch_factor=4
)

# models

In [8]:
device = "cuda" if torch.cuda.is_available() else 'cpu' 

# Pretrained components
vae = AutoencoderKL.from_pretrained(
    "stabilityai/sd-vae-ft-mse"
).to(device)
vae.requires_grad_(False)

tokenizer = CLIPTokenizer.from_pretrained(
    "openai/clip-vit-base-patch32"
)

text_encoder = CLIPTextModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)
text_encoder.requires_grad_(False)

# UNet from scratch
unet = UNet2DConditionModel(
    sample_size=8,  # 64/8 (latent space)
    in_channels=4,
    out_channels=4,
    layers_per_block=2,
    block_out_channels=(128, 256, 512, 512),
    down_block_types=(
        "DownBlock2D",
        "CrossAttnDownBlock2D",
        "CrossAttnDownBlock2D"
    ),
    up_block_types=(
        "CrossAttnUpBlock2D",
        "CrossAttnUpBlock2D",
        "UpBlock2D"
    )
).to(device)

noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-4)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


# Model training

In [9]:
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F
from tqdm import tqdm
import torch
import os

checkpoint_dir = "ldm_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

scaler = GradScaler()

steps_per_epoch = 50000
num_epochs = 3
save_every = 5000  # save every 5000 steps

global_step = 0

for epoch in range(num_epochs):

    progress_bar = tqdm(
        train_loader,
        total=steps_per_epoch,
        desc=f"Epoch {epoch}",
        leave=True
    )

    for step, (images, captions) in enumerate(progress_bar):

        if step >= steps_per_epoch:
            break

        images = images.to(device)

        # Tokenize captions
        inputs = tokenizer(
            captions,
            padding="max_length",
            truncation=True,
            max_length=77,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample()
            latents = latents * 0.18215
            text_embeddings = text_encoder(**inputs).last_hidden_state

        noise = torch.randn_like(latents)
        timesteps = torch.randint(
            0,
            noise_scheduler.num_train_timesteps,
            (latents.shape[0],),
            device=device
        ).long()

        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        with autocast():
            noise_pred = unet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=text_embeddings
            ).sample

            loss = F.mse_loss(noise_pred, noise)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        global_step += 1

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "step": global_step
        })

        # Create epoch-specific directory
        epoch_dir = os.path.join(checkpoint_dir, f"epoch_{epoch}")
        os.makedirs(epoch_dir, exist_ok=True)

        if global_step % save_every == 0:

            checkpoint_path = os.path.join(
                epoch_dir,
                f"model_epoch{epoch}_step{global_step}.pt"
            )

            torch.save({
                "epoch": epoch,
                "global_step": global_step,
                "unet_state_dict": unet.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "loss": loss.item(),
            }, checkpoint_path)

            print(f"\nSaved step checkpoint: {checkpoint_path}")
        # Save end-of-epoch model
        final_epoch_path = os.path.join(
            epoch_dir,
            f"model_epoch{epoch}.pt"
        )

        torch.save({
            "epoch": epoch,
            "global_step": global_step,
            "unet_state_dict": unet.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
        }, final_epoch_path)

    print(f"Finished epoch {epoch}")

/tmp/ipykernel_2532/1342129002.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 0:   0%|          | 0/50000 [00:00<?, ?it/s]/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/diffusers/configuration_utils.py:141: FutureWarning: Accessing config attribute `num_train_timesteps` directly via 'DDPMScheduler' object attribute is deprecated. Please access 'num_train_timesteps' over 'DDPMScheduler's config object instead, e.g. 'scheduler.config.num_train_timesteps'.

RuntimeError: mat1 and mat2 shapes cannot be multiplied (4928x512 and 1280x256)